In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/psfc.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/t2.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/SO2.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/NMVOC_finn.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/bio.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/rain.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/u10.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/swdown.npy
/kaggle/input/competitions/anrf-aise-hack-pha

In [2]:
# ═════════════════════════════════════════════════════════════════════════════
# INDIA IN THE HAZE — PHASE 2 v11 (fix1+2+3)
# ═════════════════════════════════════════════════════════════════════════════
#
# Key changes from v9 (submitted: 0.8663) and v9-fixed:
#
#  FIX-A: LOG_FEATURES restored to ALL 8 emissions + cpm25 + rain (9 total)
#         QQ plots show R²<0.07 for raw emissions; log1p is essential.
#         Rain also added: skewness 52-113, kurtosis 5000-31000.
#
#  FIX-B: Combined progressive logz weighting + episode mask boost in loss.
#         v4 had progressive (good for general high-val), v9 had episode-only.
#         Now both: graduated attention to ALL high values PLUS episode boost.
#
#  FIX-C: SMAPE eps reduced from 1.0 to 0.1. Competition denominator is
#         0.5*(|y|+|ŷ|); eps=1.0 heavily dampened low-concentration errors.
#
#  FIX-D: Removed emission scaling guard. Test data uses same WRF-Chem
#         pipeline, always in raw units. Guard risked misfire & wrong preds.
#
#  FIX-E: 2-seed ensemble. Average predictions from 2 models trained with
#         different seeds for ~0.01-0.02 free improvement.
#
#  FIX-F: Batch size 12 with gradient accumulation (effective 12) for
#         more stable gradients. Reduces wild val metric fluctuations.
#
#  Carried forward from v9-fixed:
#    - CosineAnnealingLR (better than ReduceLROnPlateau)
#    - Last-frame skip connections (not mean)
#    - Batched encoder (B*T through enc1/enc2)
#    - lat/lon static spatial channels
#    - delta_cpm25 computed locally per-window
#    - Episode mask via temporal decomposition
#    - Episode MSE boost = 3.0
#    - Temporal Gaussian smoothing (sigma=0.7)
#    - TemporalChannelAttention in decoder
# ═════════════════════════════════════════════════════════════════════════════

import os, gc, json, copy, random
import numpy as np
from typing import Dict, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim.lr_scheduler import CosineAnnealingLR
from scipy.ndimage import uniform_filter1d, gaussian_filter1d


# ═════════════════════════════════════════════════════════════════════════════
# CONFIG
# ═════════════════════════════════════════════════════════════════════════════

BASE_PATH = "/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/raw"
TEST_PATH = "/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in"
WORK_DIR  = "/kaggle/working"
os.makedirs(WORK_DIR, exist_ok=True)

MONTHS = ["APRIL_16", "JULY_16", "OCT_16", "DEC_16"]

BASE_FEATURES = [
    "cpm25", "u10", "v10", "rain", "t2", "pblh", "psfc", "swdown",
    "SO2", "NH3", "NOx", "PM25", "bio", "q2", "NMVOC_finn", "NMVOC_e",
]
DERIVED_FEATURES = ["wind_speed", "vent_index", "delta_cpm25", "lat", "lon"]
ALL_FEATURES     = BASE_FEATURES + DERIVED_FEATURES

# [FIX-A] ALL emissions + cpm25 + rain get log1p.
# QQ-plot R² proves it: SO2=0.023, NOx=0.042, PM25=0.052, NMVOC_finn=0.009,
# bio=0.274, NH3=0.319, NMVOC_e=0.184, rain=0.013. All far from normal.
LOG_FEATURES = {
    "cpm25", "SO2", "NH3", "NOx", "PM25", "bio", "NMVOC_finn", "NMVOC_e", "rain",
}

EMISSION_FEATURES = {"SO2", "NH3", "NOx", "PM25", "bio", "NMVOC_finn", "NMVOC_e"}
EMISSION_SCALE    = 1e9

PAST_HOURS   = 10
FUTURE_HOURS = 16
WINDOW       = PAST_HOURS + FUTURE_HOURS

N_TEST_SAMPLES = 218

BATCH_SIZE    = 12
ACCUM_STEPS   = 1           # effective batch = BATCH_SIZE * ACCUM_STEPS = 12
EPOCHS        = 55
WARMUP_EPOCHS = 4
PATIENCE      = 22
LR            = 3e-4
WEIGHT_DECAY  = 1e-3
HIDDEN_DIM    = 192
ENSEMBLE_SEEDS = [42, 137]  # [FIX-E] 2-seed ensemble

VAL_FRACTION    = 0.15
TEMPORAL_BUFFER = WINDOW

PM25_MAX = 400.0
LOG_CAP  = float(np.log1p(PM25_MAX))

# [FIX-B/C] Loss weights
SMAPE_WEIGHT      = 0.30
CORR_WEIGHT       = 0.15
EPISODE_MSE_BOOST = 7.0   # [FIX-3] raised from 3.0; ~5.6% ep frac needs strong boost
SMAPE_EPS         = 0.1    # [FIX-C] was 1.0

EPISODE_PERIOD = 24
TEMPORAL_SMOOTH_SIGMA = 0.7

STD_FLOOR_LOG    = 1e-4
STD_FLOOR_RAW    = 1e-4
STD_FLOOR_TARGET = 1e-4

DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()

print("=" * 80)
print("INDIA IN THE HAZE — PHASE 2 v11 (fix1+2+3)")
print(f"Device: {DEVICE}")
print(f"Ensemble seeds: {ENSEMBLE_SEEDS}")
print(f"LOG_FEATURES: {sorted(LOG_FEATURES)}")
print(f"SMAPE_EPS: {SMAPE_EPS}")
print("=" * 80)


def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)


# ═════════════════════════════════════════════════════════════════════════════
# STEP 1 — Month lengths
# ═════════════════════════════════════════════════════════════════════════════
print("\n[1/10] Reading month lengths...")
month_bounds = [0]
for month in MONTHS:
    tlen = np.load(os.path.join(BASE_PATH, month, "cpm25.npy"),
                   mmap_mode="r").shape[0]
    month_bounds.append(month_bounds[-1] + tlen)

T_TOTAL = month_bounds[-1]
print(f"  Month bounds   : {month_bounds}")
print(f"  Total timesteps: {T_TOTAL}")


# ═════════════════════════════════════════════════════════════════════════════
# STEP 2 — Train/val split (temporal, no leakage)
# ═════════════════════════════════════════════════════════════════════════════
print("\n[2/10] Computing temporal split (no leakage)...")

train_idx, val_idx = [], []
for start, end in zip(month_bounds[:-1], month_bounds[1:]):
    month_samples = list(range(start, end - WINDOW + 1))
    n = len(month_samples)
    if n < 2 * TEMPORAL_BUFFER:
        train_idx.extend(month_samples)
        continue
    cutoff = int(n * (1 - VAL_FRACTION))
    train_idx.extend(month_samples[: cutoff - TEMPORAL_BUFFER])
    val_idx.extend(month_samples[cutoff:])

train_idx = np.array(train_idx, dtype=np.int64)
val_idx   = np.array(val_idx,   dtype=np.int64)

assert len(set(train_idx.tolist()) & set(val_idx.tolist())) == 0
print(f"  Train samples : {len(train_idx)}")
print(f"  Val samples   : {len(val_idx)}")


# ═════════════════════════════════════════════════════════════════════════════
# STEP 3 — Load raw arrays + derived features
# ═════════════════════════════════════════════════════════════════════════════
print("\n[3/10] Loading raw arrays...")
raw_by_feat = {f: [] for f in BASE_FEATURES}
for month in MONTHS:
    for f in BASE_FEATURES:
        raw_by_feat[f].append(
            np.load(os.path.join(BASE_PATH, month, f"{f}.npy"), mmap_mode="r")
        )

all_data = {f: np.concatenate(raw_by_feat[f], axis=0) for f in BASE_FEATURES}
del raw_by_feat

# Derived meteorological features
all_data["wind_speed"] = np.sqrt(all_data["u10"]**2 + all_data["v10"]**2)
all_data["vent_index"] = all_data["wind_speed"] * all_data["pblh"]

# delta_cpm25: compute globally for stats, actual values computed per-window
delta = np.zeros_like(all_data["cpm25"])
delta[1:] = all_data["cpm25"][1:] - all_data["cpm25"][:-1]
for s in month_bounds[1:-1]:
    delta[s] = 0.0
all_data["delta_cpm25"] = delta

# Scale emission features by 1e9 (raw units are ~1e-11)
for f in EMISSION_FEATURES:
    all_data[f] = all_data[f] * EMISSION_SCALE

H_GRID, W_GRID = all_data["cpm25"].shape[1], all_data["cpm25"].shape[2]

# lat/lon static spatial channels
lat_lon_path = os.path.join(BASE_PATH, "lat_long.npy")
ll = np.load(lat_lon_path)
print(f"  lat_long.npy shape: {ll.shape}")

if ll.shape == (H_GRID, W_GRID, 2):
    lat_grid = ll[..., 0]
    lon_grid = ll[..., 1]
elif ll.shape == (2, H_GRID, W_GRID):
    lat_grid = ll[0]
    lon_grid = ll[1]
else:
    raise ValueError(f"Unexpected lat_long.npy shape: {ll.shape}")

def minmax_norm(x):
    lo, hi = x.min(), x.max()
    return (2.0 * (x - lo) / (hi - lo + 1e-8) - 1.0).astype(np.float32)

lat_norm = minmax_norm(lat_grid)
lon_norm = minmax_norm(lon_grid)
all_data["lat"] = np.broadcast_to(lat_norm[None], (T_TOTAL, H_GRID, W_GRID))
all_data["lon"] = np.broadcast_to(lon_norm[None], (T_TOTAL, H_GRID, W_GRID))
print(f"  lat [{lat_grid.min():.2f}, {lat_grid.max():.2f}]  "
      f"lon [{lon_grid.min():.2f}, {lon_grid.max():.2f}]")

print(f"  Shape: {all_data['cpm25'].shape}   Features: {len(ALL_FEATURES)}")


# ═════════════════════════════════════════════════════════════════════════════
# STEP 4 — Normalization stats (train-only, leakage-safe)
# ═════════════════════════════════════════════════════════════════════════════
print("\n[4/10] Computing normalization stats (train-only)...")

train_ts_mask = np.zeros(T_TOTAL, dtype=bool)
for idx in train_idx:
    train_ts_mask[idx : idx + PAST_HOURS] = True
print(f"  Train timesteps: {train_ts_mask.sum()} / {T_TOTAL}")

stats = {}
for f in ALL_FEATURES:
    arr = all_data[f]
    if arr.shape[0] == T_TOTAL:
        vals = arr[train_ts_mask].reshape(-1).astype(np.float32, copy=False)
    else:
        vals = arr.reshape(-1).astype(np.float32, copy=False)

    use_log = f in LOG_FEATURES
    if use_log:
        vals = np.log1p(np.clip(vals, 0, None)).astype(np.float32, copy=False)
    mu = float(np.nanmean(vals, dtype=np.float64))
    sd = max(float(np.nanstd(vals, dtype=np.float64)),
             STD_FLOOR_LOG if use_log else STD_FLOOR_RAW)
    stats[f] = {"mean": mu, "std": sd, "log": use_log}
    print(f"  {f:<16} {'log1p' if use_log else 'raw  '} | "
          f"mean={mu:.4e}  std={sd:.4e}")

# Target stats (log space)
sum_x = sum_x2 = 0.0; count = 0
for idx in train_idx:
    y = all_data["cpm25"][idx + PAST_HOURS : idx + PAST_HOURS + FUTURE_HOURS]
    y = np.log1p(np.clip(y, 0, None)).astype(np.float32).reshape(-1)
    sum_x  += float(y.sum(dtype=np.float64))
    sum_x2 += float((y * y).sum(dtype=np.float64))
    count  += y.size

log_mean = sum_x / max(count, 1)
var      = max(sum_x2 / max(count, 1) - log_mean**2, 0.0)
log_std  = max(float(np.sqrt(var)), STD_FLOOR_TARGET)
log_stats = {"mean": float(log_mean), "std": float(log_std)}
print(f"  log-target: mean={log_mean:.4f}  std={log_std:.4e}")

with open(os.path.join(WORK_DIR, "stats_p2.json"), "w") as fp:
    json.dump({"stats": stats, "log_stats": log_stats}, fp, indent=2)


# ═════════════════════════════════════════════════════════════════════════════
# STEP 5 — Episode mask
# ═════════════════════════════════════════════════════════════════════════════
print("\n[5/10] Computing episode mask...")


def compute_episode_mask_decomp(cpm25_month, period=24):
    T, H, W = cpm25_month.shape
    N_spatial = H * W
    data = cpm25_month.reshape(T, N_spatial).astype(np.float64)

    trend = uniform_filter1d(data, size=period, axis=0, mode="nearest")
    detrended = data - trend
    n_full = (T // period) * period
    hourly_avg = np.zeros((period, N_spatial), dtype=np.float64)
    for h in range(period):
        hourly_avg[h] = detrended[h:n_full:period].mean(axis=0)
    hourly_avg -= hourly_avg.mean(axis=0, keepdims=True)

    seasonal = np.zeros_like(data)
    for t in range(T):
        seasonal[t] = hourly_avg[t % period]

    residual = data - trend - seasonal
    res_mean = residual.mean(axis=0, keepdims=True)
    res_std  = np.maximum(residual.std(axis=0, keepdims=True), 1e-8)
    mask = (residual > res_mean + 1.5 * res_std).reshape(T, H, W)
    print(f"    Month T={T}: episode frac = {mask.mean()*100:.2f}%")
    return mask


episode_mask = np.concatenate(
    [compute_episode_mask_decomp(all_data["cpm25"][s:e], period=EPISODE_PERIOD)
     for s, e in zip(month_bounds[:-1], month_bounds[1:])],
    axis=0,
)
print(f"  Overall fraction: {episode_mask.mean()*100:.2f}%")


# ═════════════════════════════════════════════════════════════════════════════
# STEP 6 — Dataset
# ═════════════════════════════════════════════════════════════════════════════

class HazeDataset(Dataset):
    def __init__(self, data, ep_mask, stats, log_stats, past=10, future=16):
        self.data      = data
        self.ep_mask   = ep_mask
        self.stats     = stats
        self.log_stats = log_stats
        self.past      = past
        self.future    = future
        self.N         = data["cpm25"].shape[0] - (past + future) + 1

    def __len__(self):
        return self.N

    def _z(self, x, f):
        x = np.array(x, dtype=np.float32)
        if self.stats[f]["log"]:
            x = np.log1p(np.clip(x, 0, None))
        x = (x - self.stats[f]["mean"]) / self.stats[f]["std"]
        return np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

    def __getitem__(self, idx):
        past_list = []
        for f in ALL_FEATURES:
            if f == "delta_cpm25":
                cpm_w = self.data["cpm25"][idx : idx + self.past].copy().astype(np.float32)
                ld = np.zeros_like(cpm_w)
                ld[1:] = cpm_w[1:] - cpm_w[:-1]
                past_list.append(self._z(ld, f))
            else:
                past_list.append(self._z(self.data[f][idx : idx + self.past], f))

        y_raw = self.data["cpm25"][idx + self.past : idx + self.past + self.future]
        y_raw = np.clip(y_raw, 0, None).astype(np.float32)
        y     = np.log1p(y_raw)
        y     = (y - self.log_stats["mean"]) / self.log_stats["std"]
        y     = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

        ep = self.ep_mask[
            idx + self.past : idx + self.past + self.future
        ].astype(np.float32)

        return (
            torch.from_numpy(np.stack(past_list)),
            torch.from_numpy(y),
            torch.from_numpy(y_raw),
            torch.from_numpy(ep),
        )


print("\n[6/10] Building dataset/subsets...")
dataset  = HazeDataset(all_data, episode_mask, stats, log_stats,
                        PAST_HOURS, FUTURE_HOURS)
assert train_idx.max() < len(dataset)
assert val_idx.max()   < len(dataset)
train_ds = Subset(dataset, train_idx)
val_ds   = Subset(dataset, val_idx)
print(f"  Dataset={len(dataset)}  Train={len(train_ds)}  Val={len(val_ds)}")


# ═════════════════════════════════════════════════════════════════════════════
# MODEL
# ═════════════════════════════════════════════════════════════════════════════

class TemporalPosEmbedding(nn.Module):
    def __init__(self, T, C):
        super().__init__()
        self.pe = nn.Parameter(torch.randn(1, C, T, 1, 1) * 0.01)
    def forward(self, x):
        return x + self.pe


class ConvLSTMCell(nn.Module):
    def __init__(self, in_ch, hid_ch, k=3):
        super().__init__()
        self.hid_ch = hid_ch
        self.gates  = nn.Conv2d(in_ch + hid_ch, 4 * hid_ch, k, padding=k // 2)
    def forward(self, x, h, c):
        i, f, u, o = self.gates(torch.cat([x, h], 1)).chunk(4, 1)
        c2 = torch.sigmoid(f) * c + torch.sigmoid(i) * torch.tanh(u)
        return torch.sigmoid(o) * torch.tanh(c2), c2
    def init_state(self, B, H, W, dev):
        z = lambda: torch.zeros(B, self.hid_ch, H, W, device=dev)
        return z(), z()


class ConvLSTM(nn.Module):
    def __init__(self, in_ch, hid_ch, layers=3):
        super().__init__()
        self.cells = nn.ModuleList([
            ConvLSTMCell(in_ch if i == 0 else hid_ch, hid_ch)
            for i in range(layers)
        ])
    def forward(self, x):
        B, T, _, H, W = x.shape
        dev    = x.device
        states = [c.init_state(B, H, W, dev) for c in self.cells]
        all_h  = []
        for t in range(T):
            inp = x[:, t]
            for l, cell in enumerate(self.cells):
                h, c      = states[l]
                h, c      = cell(inp, h, c)
                states[l] = (h, c)
                inp = h
            all_h.append(h)
        return torch.stack(all_h, dim=1)


class TemporalSpatialAttention(nn.Module):
    def __init__(self, hid):
        super().__init__()
        self.score = nn.Conv2d(hid, 1, 1)
    def forward(self, hs):
        B, T, C, H, W = hs.shape
        s = self.score(hs.reshape(B*T, C, H, W)).reshape(B, T, 1, H, W)
        return (torch.softmax(s, 1) * hs).sum(1)


class TemporalChannelAttention(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        mid = max(ch // r, 8)
        self.mlp = nn.Sequential(
            nn.Linear(ch * 2, mid), nn.GELU(),
            nn.Linear(mid, ch), nn.Sigmoid(),
        )
    def forward(self, x):
        B, C, H, W = x.shape
        avg = x.mean(dim=[2, 3])
        mx  = x.view(B, C, -1).max(dim=2).values
        w   = self.mlp(torch.cat([avg, mx], dim=1))
        return x * w.view(B, C, 1, 1)


class SEBlock(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(ch, ch // r), nn.GELU(),
            nn.Linear(ch // r, ch), nn.Sigmoid(),
        )
    def forward(self, x):
        return x * self.fc(x).view(x.size(0), x.size(1), 1, 1)


class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, 7, padding=3)
    def forward(self, x):
        avg   = torch.mean(x, 1, keepdim=True)
        mx, _ = torch.max(x, 1, keepdim=True)
        return x * torch.sigmoid(self.conv(torch.cat([avg, mx], 1)))


class Phase2Model(nn.Module):
    def __init__(self, hidden_dim=192):
        super().__init__()
        self.C = len(ALL_FEATURES)
        self.pe = TemporalPosEmbedding(PAST_HOURS, self.C)

        self.enc1 = nn.Sequential(
            nn.Conv2d(self.C, 64, 3, padding=1), nn.BatchNorm2d(64), nn.GELU(),
            nn.Conv2d(64, 64, 3, padding=1),     nn.BatchNorm2d(64), nn.GELU(),
        )
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1),  nn.BatchNorm2d(128), nn.GELU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.GELU(),
        )
        self.pool2 = nn.MaxPool2d(2)

        self.convlstm  = ConvLSTM(128, hidden_dim, layers=3)
        self.temp_attn = TemporalSpatialAttention(hidden_dim)

        self.fuse = nn.Sequential(
            nn.Conv2d(hidden_dim * 2, 256, 3, padding=1), nn.BatchNorm2d(256), nn.GELU(),
            nn.Conv2d(256, 256, 3, padding=1),            nn.BatchNorm2d(256), nn.GELU(),
        )
        self.se  = SEBlock(256)
        self.sa  = SpatialAttention()
        self.tca = TemporalChannelAttention(256)

        self.up1  = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec1 = nn.Sequential(
            nn.Conv2d(128 + 128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.GELU(),
            nn.Conv2d(128, 128, 3, padding=1),       nn.BatchNorm2d(128), nn.GELU(),
        )
        self.up2  = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec2 = nn.Sequential(
            nn.Conv2d(64 + 64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.GELU(),
            nn.Conv2d(64, 64, 3, padding=1),      nn.BatchNorm2d(64), nn.GELU(),
        )
        self.head = nn.Conv2d(64, FUTURE_HOURS, 1)

    def forward(self, past_x):
        B, C, T, H, W = past_x.shape
        past_x = self.pe(past_x)

        # Batched encoder: (B, C, T, H, W) -> (B*T, C, H, W)
        frames = past_x.permute(0, 2, 1, 3, 4).reshape(B * T, C, H, W)

        s1_all  = self.enc1(frames)
        s2_all  = self.enc2(self.pool1(s1_all))
        seq_all = self.pool2(s2_all)

        H1, W1 = s1_all.shape[-2:]
        H2, W2 = s2_all.shape[-2:]

        s1_all = s1_all.reshape(B, T, 64,  H1, W1)
        s2_all = s2_all.reshape(B, T, 128, H2, W2)
        seq    = seq_all.reshape(B, T, 128, seq_all.shape[-2], seq_all.shape[-1])

        # Last-frame skip connections (not mean)
        skip1 = s1_all[:, -1]
        skip2 = s2_all[:, -1]

        lstm_out     = self.convlstm(seq)
        attn_summary = self.temp_attn(lstm_out)
        last_frame   = lstm_out[:, -1]

        x = self.fuse(torch.cat([attn_summary, last_frame], dim=1))
        x = self.tca(self.sa(self.se(x)))

        up1_out = self.up1(x)
        if up1_out.shape[-2:] != skip2.shape[-2:]:
            up1_out = F.interpolate(up1_out, skip2.shape[-2:],
                                    mode="bilinear", align_corners=False)
        x = self.dec1(torch.cat([up1_out, skip2], 1))

        up2_out = self.up2(x)
        if up2_out.shape[-2:] != skip1.shape[-2:]:
            up2_out = F.interpolate(up2_out, skip1.shape[-2:],
                                    mode="bilinear", align_corners=False)
        x = self.dec2(torch.cat([up2_out, skip1], 1))

        if x.shape[-2:] != (H, W):
            x = F.interpolate(x, (H, W), mode="bilinear", align_corners=False)
        return self.head(x)


# ═════════════════════════════════════════════════════════════════════════════
# Loss and metrics
# ═════════════════════════════════════════════════════════════════════════════

def smape(pred, target, eps=SMAPE_EPS):
    """[FIX-C] eps reduced from 1.0 to 0.1"""
    return (2.0 * torch.abs(pred - target) /
            (torch.abs(pred) + torch.abs(target) + eps)).mean()


def spatial_pearson(pred, target):
    B, T, H, W = pred.shape
    p = pred.reshape(B, T, -1)
    t = target.reshape(B, T, -1)
    p = p - p.mean(dim=-1, keepdim=True)
    t = t - t.mean(dim=-1, keepdim=True)
    num = (p * t).sum(dim=-1)
    den = p.norm(dim=-1) * t.norm(dim=-1) + 1e-8
    return (num / den).mean()


def episode_aware_loss(pred_logz, target_logz, target_raw, ep_mask, epoch,
                       log_stats_dict, warmup=WARMUP_EPOCHS):
    """
    [FIX-B] Combined progressive logz weighting + episode mask boost.
    Progressive weights give graduated attention to ALL high values.
    Episode mask adds extra boost specifically to statistically-defined episodes.
    """
    # Progressive weighting based on target magnitude in log space
    y_log = target_logz * log_stats_dict["std"] + log_stats_dict["mean"]
    w = (1.0
         + 0.5 * (y_log > 1.0).float()
         + 1.0 * (y_log > 2.0).float()
         + 1.0 * (y_log > 3.0).float()
         + 1.5 * (y_log > 4.0).float()
         + 2.0 * (y_log > 5.0).float())

    # Episode boost on top
    w = w + EPISODE_MSE_BOOST * ep_mask

    mse = torch.sqrt(
        (w * (pred_logz - target_logz) ** 2).sum() / (w.sum() + 1e-8) + 1e-8
    )

    loss_smape = torch.tensor(0.0, device=pred_logz.device)
    loss_corr  = torch.tensor(0.0, device=pred_logz.device)

    if epoch >= warmup:
        pred_log  = pred_logz * log_stats_dict["std"] + log_stats_dict["mean"]
        pred_log  = torch.clamp(pred_log, 0.0, LOG_CAP)
        pred_real = torch.expm1(pred_log).clamp(min=0.0)
        alpha = min(1.0, (epoch - warmup + 1) / 5.0)
        loss_smape = alpha * SMAPE_WEIGHT * smape(pred_real, target_raw)
        loss_corr  = alpha * CORR_WEIGHT  * (1.0 - spatial_pearson(pred_real, target_raw))

    # [FIX-3] Dedicated episodic MAE — less dominated by easy low-conc pixels
    # than MSE, directly penalises peak magnitude errors that drive epSMAPE
    ep_count = ep_mask.sum() + 1e-8
    ep_mae   = (torch.abs(pred_logz - target_logz) * ep_mask).sum() / ep_count
    return mse + 0.5 * ep_mae + loss_smape + loss_corr, mse


def compute_episodic_metrics(p_real, t_raw, idx_list):
    eps = SMAPE_EPS
    smape_t, corr_t = [], []
    total_ep = total_all = 0

    for t_step in range(FUTURE_HOURS):
        pred_list, tar_list = [], []
        for i, sample_idx in enumerate(idx_list):
            ts = int(sample_idx) + PAST_HOURS + t_step
            if ts >= T_TOTAL:
                continue
            ep = episode_mask[ts]
            if not ep.any():
                continue
            ep_t = torch.from_numpy(ep).to(p_real.device)
            pred_list.append(p_real[i, t_step][ep_t])
            tar_list.append(t_raw[i, t_step][ep_t])
            total_ep += ep_t.sum().item()
        total_all += len(idx_list) * H_GRID * W_GRID
        if not pred_list:
            continue
        p = torch.cat(pred_list); y = torch.cat(tar_list)
        sm = (2.0 * torch.abs(p - y) / (torch.abs(p) + torch.abs(y) + eps)).mean()
        smape_t.append(sm)
        pc = p - p.mean(); yc = y - y.mean()
        corr_t.append((pc * yc).sum() / (pc.norm() * yc.norm() + 1e-8))

    if not smape_t:
        return float("nan"), float("nan"), 0.0
    return (torch.stack(smape_t).mean().item(),
            torch.stack(corr_t).mean().item(),
            total_ep / max(total_all, 1))


def eval_metrics(model, loader, log_stats_dict, idx_list=None):
    model.eval()
    all_pred, all_tar, all_raw = [], [], []
    with torch.no_grad():
        for px, y, y_raw, _ep in loader:
            pred = model(px.to(DEVICE)).cpu()
            all_pred.append(pred); all_tar.append(y); all_raw.append(y_raw)

    p     = torch.cat(all_pred, 0)
    t     = torch.cat(all_tar,  0)
    t_raw = torch.cat(all_raw,  0)

    p_log  = torch.clamp(p * log_stats_dict["std"] + log_stats_dict["mean"],
                         0.0, LOG_CAP)
    p_real = torch.expm1(p_log).clamp(min=0.0, max=PM25_MAX)

    rmse_real = torch.sqrt(((p_real - t_raw) ** 2).mean()).item()
    smape_val = smape(p_real, t_raw).item()
    corr_val  = spatial_pearson(p_real, t_raw).item()

    ep_smape = ep_corr = float("nan"); ep_frac = 0.0
    if idx_list is not None:
        ep_smape, ep_corr, ep_frac = compute_episodic_metrics(
            p_real, t_raw, idx_list)

    return {
        "rmse_real": rmse_real, "smape": smape_val, "corr": corr_val,
        "ep_smape": ep_smape,   "ep_corr": ep_corr, "ep_frac": ep_frac,
    }


# ═════════════════════════════════════════════════════════════════════════════
# Training function (called per seed)
# ═════════════════════════════════════════════════════════════════════════════

def train_one_seed(seed):
    print(f"\n{'='*60}")
    print(f"  TRAINING SEED {seed}")
    print(f"{'='*60}")
    set_seed(seed)

    g = torch.Generator(); g.manual_seed(seed)
    tr_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=2, pin_memory=True, generator=g)
    va_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=2, pin_memory=True)

    model = Phase2Model(hidden_dim=HIDDEN_DIM).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Model parameters: {n_params:,}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR,
                                  weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    scaler    = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    best_score = -1e9
    best_w     = copy.deepcopy(model.state_dict())
    pat        = 0

    for ep in range(EPOCHS):
        model.train()
        run_mse = 0.0
        optimizer.zero_grad(set_to_none=True)

        for step, (px, y, y_raw, ep_mask_batch) in enumerate(tr_loader):
            px     = px.to(DEVICE)
            y      = y.to(DEVICE)
            y_raw  = y_raw.to(DEVICE)
            ep_mb  = ep_mask_batch.to(DEVICE)

            with torch.amp.autocast("cuda", enabled=USE_AMP):
                pred = model(px)
                loss, mse = episode_aware_loss(
                    pred, y, y_raw, ep_mb, ep, log_stats_dict=log_stats)

            # Gradient accumulation
            loss_scaled = loss / ACCUM_STEPS
            scaler.scale(loss_scaled).backward()

            if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(tr_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            run_mse += mse.item()

        scheduler.step()
        tr_mse = run_mse / max(len(tr_loader), 1)
        vm     = eval_metrics(model, va_loader, log_stats_dict=log_stats,
                              idx_list=val_idx)

        # [FIX-2] Competition weights undisclosed — use epSMAPE as primary
        # criterion (our weakest axis). Secondary penalty on gSMAPE, reward
        # for epCorr. Negated so "higher = better" convention is preserved.
        if not (np.isnan(vm["ep_smape"]) or np.isnan(vm["ep_corr"])):
            proxy_score = -(vm["ep_smape"] + 0.5 * vm["smape"] - 0.3 * vm["ep_corr"])
        else:
            proxy_score = -(vm["smape"] - 0.5 * vm["corr"])

        is_best = proxy_score > best_score
        if is_best:
            best_score = proxy_score
            best_w     = copy.deepcopy(model.state_dict())
            pat        = 0
        else:
            pat += 1

        lr_now = optimizer.param_groups[0]["lr"]
        print(
            f"  Ep {ep+1:02d} | tr_mse {tr_mse:.4f} | "
            f"score {proxy_score:.4f} | "
            f"gSMAPE {vm['smape']:.4f} | epSMAPE {vm['ep_smape']:.4f} | "
            f"epCorr {vm['ep_corr']:.4f} | rmse {vm['rmse_real']:.2f} | "
            f"lr {lr_now:.2e}"
            f" {'*' if is_best else ''}"
        )

        if pat >= PATIENCE:
            print(f"  Early stop ep {ep+1} — best: {best_score:.4f}")
            break

    model.load_state_dict(best_w)
    vm_final = eval_metrics(model, va_loader, log_stats_dict=log_stats,
                            idx_list=val_idx)
    print(f"\n  Seed {seed} best metrics:")
    print(f"    RMSE(real)    : {vm_final['rmse_real']:.2f}")
    print(f"    Global SMAPE  : {vm_final['smape']:.4f}")
    print(f"    Episode SMAPE : {vm_final['ep_smape']:.4f}")
    print(f"    Episode Corr  : {vm_final['ep_corr']:.4f}")

    return model, vm_final, best_score


# ═════════════════════════════════════════════════════════════════════════════
# STEP 7 — Training (ensemble)
# ═════════════════════════════════════════════════════════════════════════════
print("\n[7/10] Training ensemble...")

models = []
all_vm = []
for seed in ENSEMBLE_SEEDS:
    model, vm_final, best_score = train_one_seed(seed)
    models.append(model)
    all_vm.append(vm_final)
    gc.collect()
    torch.cuda.empty_cache()


# ═════════════════════════════════════════════════════════════════════════════
# STEP 8-10 — Inference
# ═════════════════════════════════════════════════════════════════════════════
print("\n[8/10] Loading test arrays...")
test_arrays = {}
for f in BASE_FEATURES:
    test_arrays[f] = np.load(os.path.join(TEST_PATH, f"{f}.npy"), mmap_mode="r")
    print(f"  {f:<12} shape={test_arrays[f].shape}")

test_arrays["wind_speed"] = np.sqrt(test_arrays["u10"]**2 + test_arrays["v10"]**2)
test_arrays["vent_index"] = test_arrays["wind_speed"] * test_arrays["pblh"]

delta_t = np.zeros_like(test_arrays["cpm25"])
delta_t[:, 1:] = test_arrays["cpm25"][:, 1:] - test_arrays["cpm25"][:, :-1]
test_arrays["delta_cpm25"] = delta_t

# lat/lon for test
lat_t = np.tile(lat_norm[None, None], (N_TEST_SAMPLES, PAST_HOURS, 1, 1))
lon_t = np.tile(lon_norm[None, None], (N_TEST_SAMPLES, PAST_HOURS, 1, 1))
test_arrays["lat"] = lat_t
test_arrays["lon"] = lon_t


def build_test_sample(i):
    """Build a single test sample. [FIX-D] Always apply EMISSION_SCALE."""
    past_list = []
    for f in ALL_FEATURES:
        arr = test_arrays[f][i]
        vals = arr[:PAST_HOURS].copy().astype(np.float32)
        # [FIX-D] Always scale emissions — no guard heuristic
        if f in EMISSION_FEATURES:
            vals = vals * EMISSION_SCALE
        if stats[f]["log"]:
            vals = np.log1p(np.clip(vals, 0, None))
        vals = (vals - stats[f]["mean"]) / stats[f]["std"]
        vals = np.nan_to_num(vals, nan=0.0, posinf=0.0, neginf=0.0)
        past_list.append(vals)
    return torch.tensor(np.stack(past_list), dtype=torch.float32).unsqueeze(0)


def temporal_gaussian_smooth(pred, sigma=TEMPORAL_SMOOTH_SIGMA):
    if sigma <= 0.0:
        return pred
    return gaussian_filter1d(pred, sigma=sigma, axis=-1,
                             mode="nearest").astype(np.float32)


print("\n[9/10] Running ensemble inference...")
ensemble_preds = []

for m_idx, model in enumerate(models):
    model.eval()
    model_preds = []
    with torch.no_grad():
        for i in range(N_TEST_SAMPLES):
            px = build_test_sample(i).to(DEVICE)
            o  = model(px)
            o_log  = o * log_stats["std"] + log_stats["mean"]
            o_log  = torch.clamp(o_log, min=0.0, max=LOG_CAP)
            o_real = torch.clamp(torch.expm1(o_log), min=0.0, max=PM25_MAX)
            model_preds.append(o_real.cpu().numpy())
            if i % 100 == 0:
                print(f"  Model {m_idx+1}: processed {i}/{N_TEST_SAMPLES}")

    pred_m = np.concatenate(model_preds, axis=0)       # (N, T, H, W)
    pred_m = np.transpose(pred_m, (0, 2, 3, 1))        # (N, H, W, T)
    ensemble_preds.append(pred_m)
    print(f"  Model {m_idx+1}: pred range [{pred_m.min():.1f}, {pred_m.max():.1f}], "
          f"mean={pred_m.mean():.1f}")

# [FIX-E] Average ensemble predictions
pred = np.mean(ensemble_preds, axis=0)

# [FIX-1] Temporal smoothing REMOVED — Gaussian blurs episode peaks,
# directly hurting epSMAPE and epCorr. Coherence enforced via training loss instead.

pred = np.nan_to_num(pred, nan=0.0).astype(np.float32)

print(f"\n  Final prediction stats:")
print(f"    shape : {pred.shape}")
print(f"    min   : {pred.min():.3f}")
print(f"    max   : {pred.max():.3f}  (hard cap: {PM25_MAX})")
print(f"    mean  : {pred.mean():.3f}")

assert pred.shape == (N_TEST_SAMPLES, H_GRID, W_GRID, FUTURE_HOURS), \
    f"Shape error: got {pred.shape}"
print("  Shape check : PASSED")


# ═════════════════════════════════════════════════════════════════════════════
# Save
# ═════════════════════════════════════════════════════════════════════════════
print("\n[10/10] Saving predictions...")
np.save(os.path.join(WORK_DIR, "preds.npy"), pred)
print("  Saved: /kaggle/working/preds.npy")

summary = {
    "version"           : "phase2_v11-fix123",
    "ensemble_seeds"    : ENSEMBLE_SEEDS,
    "n_features"        : len(ALL_FEATURES),
    "feature_list"      : ALL_FEATURES,
    "log_features"      : sorted(LOG_FEATURES),
    "smape_eps"         : SMAPE_EPS,
    "episode_boost"     : EPISODE_MSE_BOOST,
    "temporal_smooth"   : TEMPORAL_SMOOTH_SIGMA,
    "pred_min"          : float(pred.min()),
    "pred_max"          : float(pred.max()),
    "pred_mean"         : float(pred.mean()),
    "per_seed_metrics"  : [
        {k: float(v) if not isinstance(v, float) else v
         for k, v in vm.items()} for vm in all_vm
    ],
    "key_fixes": [
        "FIX-A: LOG_FEATURES = all 8 emissions + cpm25 + rain",
        "FIX-B: Combined progressive logz weighting + episode mask boost",
        "FIX-C: SMAPE eps reduced from 1.0 to 0.1",
        "FIX-D: Removed emission scaling guard, always apply EMISSION_SCALE",
        "FIX-E: 2-seed ensemble (average predictions)",
        "FIX-F: Batch size 12",
        "FIX-1: Removed post-inference Gaussian smoothing (blurs episode peaks)",
        "FIX-2: Proxy score = -(epSMAPE + 0.5*gSMAPE - 0.3*epCorr), epSMAPE primary",
        "FIX-3: EPISODE_MSE_BOOST=7.0 + dedicated episodic MAE term (0.5 weight)",
        "Carried: CosineAnnealingLR, last-frame skips, batched encoder",
        "Carried: lat/lon, delta_cpm25, episode decomposition, TCA",
    ],
}
with open(os.path.join(WORK_DIR, "run_summary_p2.json"), "w") as fp:
    json.dump(summary, fp, indent=2)

print("\n" + "=" * 80)
print("DONE — PHASE 2 v11 (fix1+2+3)")
print(f"  Ensemble seeds       : {ENSEMBLE_SEEDS}")
print(f"  LOG_FEATURES         : {sorted(LOG_FEATURES)}")
print(f"  SMAPE eps            : {SMAPE_EPS}")
print(f"  Episode MSE boost    : {EPISODE_MSE_BOOST}")
for i, vm in enumerate(all_vm):
    print(f"  Seed {ENSEMBLE_SEEDS[i]} — gSMAPE={vm['smape']:.4f}  "
          f"epSMAPE={vm['ep_smape']:.4f}  epCorr={vm['ep_corr']:.4f}")
print(f"  Pred: [{pred.min():.1f}, {pred.max():.1f}], mean={pred.mean():.1f}")
print(f"  Submit: /kaggle/working/preds.npy  shape={pred.shape}")
print("=" * 80)

INDIA IN THE HAZE — PHASE 2 v11 (fix1+2+3)
Device: cuda
Ensemble seeds: [42, 137]
LOG_FEATURES: ['NH3', 'NMVOC_e', 'NMVOC_finn', 'NOx', 'PM25', 'SO2', 'bio', 'cpm25', 'rain']
SMAPE_EPS: 0.1

[1/10] Reading month lengths...
  Month bounds   : [0, 715, 1454, 2193, 2932]
  Total timesteps: 2932

[2/10] Computing temporal split (no leakage)...
  Train samples : 2300
  Val samples   : 428

[3/10] Loading raw arrays...
  lat_long.npy shape: (140, 124, 2)
  lat [7.06, 38.00]  lon [67.79, 97.93]
  Shape: (2932, 140, 124)   Features: 21

[4/10] Computing normalization stats (train-only)...
  Train timesteps: 2336 / 2932
  cpm25            log1p | mean=2.8360e+00  std=1.1786e+00
  u10              raw   | mean=1.7290e+00  std=3.5530e+00
  v10              raw   | mean=2.0097e-01  std=3.0438e+00
  rain             log1p | mean=4.6735e-02  std=2.1635e-01
  t2               raw   | mean=2.9181e+02  std=1.3840e+01
  pblh             raw   | mean=7.5694e+02  std=6.0789e+02
  psfc             raw   | 